# Analysis Code for Entity Impact Prediction

This notebook contains custom analysis and visualization code developed for the Entity Impact Prediction project.

**Authors**: Our own implementation
**Purpose**: Result analysis and visualization of entity impact prediction outcomes.


Deduplication

In [ ]:
import pandas as pd
import os

def remove_duplicate_entities(file_path):
    # Read CSV file
    df = pd.read_csv(file_path, sep=',')
    
    # Clean column names
    df.columns = df.columns.str.strip()
    
    # Create case-insensitive entity name column
    df['..._...'] = df['...'].str.lower()
    
    # Remove duplicates based on lowercase entity name, keep first occurrence
    df_deduplicated = df.drop_duplicates(subset=['..._...'], keep='first')
    
    # Remove temporary column
    df_deduplicated = df_deduplicated.drop(columns=['..._...'])
    
    # Save processed file
    output_path = file_path.replace('.csv', '_deduplicated.csv')
    df_deduplicated.to_csv(output_path, sep=',', index=False, encoding='utf-8')
    
    print(f"Processing complete...Original data: {len(df)} rows...After deduplication: {len(df_deduplicated)} rows")
    print(f"...Save to: {output_path}")

if __name__ == "__main__":
    file_path = r"./data/entities\extracted_entities.csv"
    remove_duplicate_entities(file_path)

Remove parentheses

In [ ]:
import csv
import re

def clean_entities(input_file, output_file):
    """
    ...Delete...
    """
    cleaned_entities = []
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)  # Read header
        
        for row in reader:
            if len(row) >= 2:
                entity_name = row[1].strip()
                # Remove parentheses and their contents (supports English parentheses)
                # Match all parentheses and their contents, including nested
                cleaned_name = re.sub(r'\s*\([^)]*\)', '', entity_name)
                cleaned_name = cleaned_name.strip()
                
                if cleaned_name:  # Only keep non-empty entities
                    cleaned_entities.append([row[0], cleaned_name, row[2], row[3], row[4]])
    
    # Save cleaned file
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['...ID', '...', '...', 'Start...', 'End...'])
        writer.writerows(cleaned_entities)
    
    print(f"...: {len(cleaned_entities)}")
    print(f"...Save to: {output_file}")

if __name__ == "__main__":
    input_file = r"./data/entities\extracted_entities_deduplicated.csv"
    output_file = r"./data/entities\extracted_entities_cleaned.csv"
    
    clean_entities(input_file, output_file)

Clean plural forms

In [ ]:
import csv
import re

def is_plural(word):
    """
     [description] 
    ...
    1. ...s...
    2. ...ss, sss...process, class...
    3.  [description] 
    4. ... columns...
    """
    if not word.endswith('s'):
        return False
    
    # Code comment
    if word.endswith('ss') or word.endswith('sss'):
        return False
    
    # Code comment
    # Code comment
    if word[:-1].isupper() and word[-1] == 's':
        return True
    
    # Code comment
    if any(c.isdigit() for c in word):
        return False
    
    return True

def get_singular(word):
    """
    ...
    """
    if not is_plural(word):
        return word
    
    # Handle plurals ending with s
    if word.endswith('ies'):
        return word[:-3] + 'y'
    elif word.endswith('es'):
        return word[:-2]
    elif word.endswith('s'):
        return word[:-1]
    
    return word

def clean_plural_entities(input_file, output_file):
    """
    Delete [description] 
    """
    # Read all entities
    entities = []
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)
        for row in reader:
            if len(row) >= 2:
                entities.append(row)
    
    # Build entity name set
    entity_names = [row[1] for row in entities]
    entity_set = set(entity_names)
    
    # Find entities to keep (prioritize singular forms)
    keep_indices = []
    remove_names = set()
    
    for i, row in enumerate(entities):
        entity_name = row[1]
        
        # Skip if already marked for deletion
        if entity_name in remove_names:
            continue
        
        # Code comment
        if is_plural(entity_name):
            singular = get_singular(entity_name)
            
            # Code comment
            if singular in entity_set:
                remove_names.add(entity_name)
                continue
            else:
                # Code comment
                keep_indices.append(i)
        else:
            # Code comment
            keep_indices.append(i)
    
      # Comment
    cleaned_entities = [entities[i] for i in keep_indices]
    
    # Save cleaned file
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(cleaned_entities)
    
    print(f"...: {len(entities)}")
    print(f"Delete...: {len(remove_names)}")
    print(f"...: {len(cleaned_entities)}")
    print(f"\nDelete...:")
    for name in list(remove_names)[:20]:
        print(f"  {name} -> {get_singular(name)}")
    
    return cleaned_entities

def main():
    input_file = r"./data/entities\extracted_entities_cleaned.csv"
    output_file = r"./data/entities\extracted_entities_singular.csv"
    
    cleaned_entities = clean_plural_entities(input_file, output_file)
    print(f"\n...Done...ResultSave to: {output_file}")

if __name__ == "__main__":
    main()

Delete

In [ ]:
import csv
import re

def clean_special_characters(input_file, output_file):
    """
    Delete [description] : ~ " $ % \ /
    ...: & . - + '
    """
    # Code comment
    chars_to_remove = set('~"$%\\/')
    
    # Read all entities
    cleaned_entities = []
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)
        
        for row in reader:
            if len(row) >= 5:
                entity_name = row[1]
                
                  # Comment
                cleaned_name = ''.join(char for char in entity_name if char not in chars_to_remove)
                
                  # Comment
                cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip()
                
                if cleaned_name:  # Only keep non-empty entities
                    row[1] = cleaned_name
                    cleaned_entities.append(row)
    
    # Save cleaned file
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(cleaned_entities)
    
    print(f"...: {len(cleaned_entities)}")
    print(f"...Save to: {output_file}")
    
    

def main():
    input_file = r"./data/entities\extracted_entities_singular.csv"
    output_file = r"./data/entities\extracted_entities_cleaned_final.csv"
    
    clean_special_characters(input_file, output_file)
    print("\nProcessing complete...")

if __name__ == "__main__":
    main()

Statistics

In [ ]:
import sys
!{sys.executable} -m pip install pyahocorasick

In [ ]:
# Code comment
import sys
!{sys.executable} -m pip install pyahocorasick

# Code comment
from ahocorasick import Automaton

In [ ]:
!pip install pyahocorasick

Statistics [description] columns

In [ ]:
import csv
import json
from collections import defaultdict
from tqdm import tqdm
import sys
import os
import re
from ahocorasick import Automaton

def read_entities(file_path):
    """ReadDeduplication...CSV [description] Size..."""
    print(f"Reading entities from: {file_path}")
    
    entities = []
    # Code comment
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'latin-1', 'cp1252']
    
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as file:
                reader = csv.reader(file)
                header = next(reader)
                
                for row in reader:
                    if len(row) >= 2:
                        entity_name = row[1].strip()
                        if entity_name:
                            entities.append(entity_name)
            
            print(f"Successfully read with encoding: {enc}")
            print(f"Loaded {len(entities)} unique entities.")
            print(f"Sample entities: {entities[:5]}")
            return entities
            
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise Exception("Failed to read entities file with any encoding")

def read_papers(file_path):
    """Read...JSON..."""
    print(f"Reading papers from: {file_path}")
    
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030']
    
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as file:
                papers = json.load(file)
            print(f"Successfully read with encoding: {enc}")
            print(f"Loaded {len(papers)} papers.")
            return papers
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    
    raise Exception("Failed to read papers file with any encoding")

def count_entity_document_frequency_ac(entities, papers):
    """
    ...AC...Statistics [description] DF...
    ...Size [description] 
    """
    print("Building AC automaton...")
    
      # Comment
    automaton = Automaton()
    entity_lengths = {}
    
      # Comment
    for entity in entities:
        automaton.add_word(entity, entity)  # Code comment
        entity_lengths[entity] = len(entity)
    
      # Comment
    automaton.make_automaton()
    print(f"AC automaton built with {len(entities)} patterns.")
    
    # Code comment
    word_boundaries = set(" ,.;:!?()[]{}\"'-\n\t\r")
    
    print("Counting entity document frequency...")
    
      # Comment
    entity_docs = defaultdict(set)
    total_papers = len(papers)
    
    # Code comment
    for idx, paper in tqdm(enumerate(papers, 1), total=total_papers, desc="Processing papers", ncols=100):
          # Comment
        title = paper.get("title", "")
        abstract = paper.get("abstract", "")
        text = title + " " + abstract  # Code comment
        
        # Code comment
        matched_entities = set()
        
        # Code comment
        for end_index, entity in automaton.iter(text):
            entity_len = entity_lengths[entity]
            start_index = end_index - entity_len + 1
            
            # Code comment
              # Comment
            if start_index > 0 and text[start_index - 1] not in word_boundaries:
                continue
              # Comment
            if end_index + 1 < len(text) and text[end_index + 1] not in word_boundaries:
                continue
            
            # Code comment
            matched_entities.add(entity)
        
        # Code comment
        for entity in matched_entities:
            entity_docs[entity].add(idx)
    
    print(f"Counting complete. Processed {total_papers} papers.")
    
      # Comment
    entity_df = {}
    for entity, doc_set in entity_docs.items():
        entity_df[entity] = len(doc_set)
    
    total_matches = sum(entity_df.values())
    entities_with_matches = sum(1 for freq in entity_df.values() if freq > 0)
    print(f"Total document-entity pairs: {total_matches}")
    print(f"Entities with matches: {entities_with_matches}/{len(entities)}")
    
    return entity_df

def save_results_with_percentage(results, output_file_path, total_docs):
    """
    SaveStatisticsResult [description] 
    Delete...1...Sort
    """
      # Comment
    filtered_results = {entity: freq for entity, freq in results.items() if freq >= 1}
    
    # Code comment
    sorted_results = sorted(filtered_results.items(), key=lambda x: x[1], reverse=True)
    
      # Comment
    total_freq_sum = sum(filtered_results.values())
    
    print(f"\nStatistics after filtering:")
    print(f"Entities with frequency >= 1: {len(filtered_results)}")
    print(f"Total frequency sum: {total_freq_sum}")
    
      # Comment
    output_csv = output_file_path.replace('.txt', '.csv')
    
    # Code comment
    output_dir = os.path.dirname(output_csv)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    try:
        with open(output_csv, 'w', encoding='utf-8-sig', newline='') as file:
            writer = csv.writer(file)
              # Comment
            writer.writerow(['Entity Name', 'Document Frequency', 'Percentage (%)', 'Cumulative Percentage (%)'])
            
              # Comment
            cumulative_percentage = 0.0
            
              # Comment
            for entity, freq in sorted_results:
                try:
                      # Comment
                    percentage = (freq / total_freq_sum) * 100
                    # Code comment
                    cumulative_percentage += percentage
                    # Write rows
                    writer.writerow([entity, freq, f"{percentage:.4f}%", f"{cumulative_percentage:.4f}%"])
                except Exception as e:
                    print(f"Error writing entity '{entity}': {e}")
                    continue
        
        print(f"\nResults saved to: {output_csv}")
        
    except Exception as e:
        print(f"Error saving CSV file: {e}")
    
    # Code comment
    try:
        with open(output_file_path, 'w', encoding='utf-8') as file:
            for entity, freq in sorted_results:
                file.write(f"{entity}\t{freq}\n")
        print(f"TXT results saved to: {output_file_path}")
    except Exception as e:
        print(f"Error saving TXT file: {e}")
    
    return filtered_results, total_freq_sum

def main():
    """..."""
    print("=" * 60)
    print("Entity Document Frequency Counter (AC Automaton Version)")
    print("=" * 60)
    
    # Code comment
    entity_file = r"./data/entities\extracted_entities_cleaned.csv"
    paper_file = r"./data/arxiv_cs_original.json"
    output_file = r"./data/entities\entity_document_frequency.txt"
    
    # Code comment
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Code comment
    if not os.path.exists(entity_file):
        print(f"Deduplication...does not exist......")
        entity_file = r"./data/entities\extracted_entities.csv"
    
      # Comment
    if not os.path.exists(entity_file):
        print(f"Error: ...does not exist: {entity_file}")
        return
    
    if not os.path.exists(paper_file):
        print(f"Error: ...does not exist: {paper_file}")
        return
    
    # Read data
    entities = read_entities(entity_file)
    papers = read_papers(paper_file)
    
    print(f"\nStarting document frequency counting...")
    print(f"Unique entities: {len(entities)}, Papers: {len(papers)}")
    
      # Comment
    entity_df = count_entity_document_frequency_ac(entities, papers)
    
    # SaveResult
    filtered_results, total_freq_sum = save_results_with_percentage(
        entity_df, output_file, len(papers)
    )
    
      # Comment
    if filtered_results:
        print("\nTop 10 entities by document frequency:")
        sorted_items = sorted(filtered_results.items(), key=lambda x: x[1], reverse=True)[:10]
        cumulative = 0.0
        for entity, freq in sorted_items:
            percentage = (freq / total_freq_sum) * 100
            cumulative += percentage
            print(f"  {entity}: {freq} ({percentage:.4f}%, Cumulative: {cumulative:.4f}%)")
    
    print("\nProcessing complete...")

if __name__ == "__main__":
    main()

## Processing Step

Analysis and data processing code for Entity Impact Prediction project.

In [ ]:
import pandas as pd

  # Comment
entities_df = pd.read_csv(r"./data/entities\extracted_entities_cleaned_final.csv", encoding='utf-8-sig')
predictions_df = pd.read_csv(r"./data/entities\results\all_predictions_with_entities.csv", encoding='utf-8-sig')

  # Comment
entity_to_category = dict(zip(entities_df['...'], entities_df['...']))

# Code comment
predictions_df['concept1_category'] = predictions_df['concept1_keyword'].map(entity_to_category)
predictions_df['concept2_category'] = predictions_df['concept2_keyword'].map(entity_to_category)

# SaveResult
output_path = r"./data/entities\results\all_predictions_with_categories.csv"
predictions_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Processing complete...Result...Save to: {output_path}")
print(f"...Process {len(predictions_df)} ...")

Statistics

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn

In [ ]:
import pandas as pd

df = pd.read_csv(r"./data/entities\extracted_entities_cleaned_final.csv", encoding='utf-8-sig')

result = df['...'].value_counts().to_frame('Count')
result['Percentage (%)'] = df['...'].value_counts(normalize=True).mul(100).round(2)
print(result)
print(f"\nTotal: {len(df)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

# Read data
file_path = r"./data/entities\results\all_predictions_with_categories.csv"
df = pd.read_csv(file_path, encoding='utf-8-sig')

# Get categories
categories = sorted(set(df['concept1_category'].unique()) | set(df['concept2_category'].unique()))

# Create symmetric category combination matrix
def create_symmetric_category_matrix(df):
    categories = sorted(set(df['concept1_category'].unique()) | set(df['concept2_category'].unique()))
    
    matrix_data = []
    for i, cat1 in enumerate(categories):
        row = []
        for j, cat2 in enumerate(categories):
            if i <= j:
                subset1 = df[(df['concept1_category'] == cat1) & (df['concept2_category'] == cat2)]
                subset2 = df[(df['concept1_category'] == cat2) & (df['concept2_category'] == cat1)]
                subset = pd.concat([subset1, subset2])
                
                if len(subset) > 0:
                    avg_score = subset['score'].mean()
                    count = len(subset)
                    std_score = subset['score'].std() if len(subset) > 1 else 0
                else:
                    avg_score = np.nan
                    count = 0
                    std_score = 0
                
                row.append({'avg_score': avg_score, 'count': count, 'std': std_score})
            else:
                row.append(matrix_data[j][i])
        matrix_data.append(row)
    
    return categories, matrix_data

categories, matrix_data = create_symmetric_category_matrix(df)

avg_score_matrix = pd.DataFrame(
    [[cell['avg_score'] for cell in row] for row in matrix_data],
    index=categories, columns=categories
)

# ==================== Average Score Heatmap (Separate) ====================
# Create figure for average score heatmap only
fig, ax = plt.subplots(figsize=(8, 6))

# Calculate color range based on data distribution
vmin = avg_score_matrix.values.min()
vmax = avg_score_matrix.values.max()
# Add small padding to make colors more sensitive
vmin_padded = vmin - (vmax - vmin) * 0.05
vmax_padded = vmax + (vmax - vmin) * 0.05

# Use a more sensitive colormap with more color gradations
im = ax.imshow(avg_score_matrix.values, cmap='RdYlGn_r', aspect='auto', 
               vmin=vmin_padded, vmax=vmax_padded, interpolation='bilinear')

# Set ticks and labels
ax.set_xticks(range(len(categories)))
ax.set_yticks(range(len(categories)))
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=12)
ax.set_yticklabels(categories, fontsize=12)
ax.set_title('Average Score Matrix (Symmetric)', fontsize=16, fontweight='bold')
ax.set_xlabel('Category', fontsize=13)
ax.set_ylabel('Category', fontsize=13)

# Add text annotations with more precision
for i in range(len(categories)):
    for j in range(len(categories)):
        if not np.isnan(avg_score_matrix.values[i, j]):
            # Determine text color based on background
            value = avg_score_matrix.values[i, j]
            # Normalize value for color brightness calculation
            norm_value = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            # Use white text for dark backgrounds, black for light backgrounds
            if norm_value > 0.6:
                text_color = 'white'
            else:
                text_color = 'black'
            ax.text(j, i, f'{value:.6f}', ha="center", va="center", 
                   color=text_color, fontsize=10, fontweight='bold')

# Add colorbar with more ticks for better sensitivity
cbar = plt.colorbar(im, ax=ax, label='Average Score', fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=10)

# Save the figure
output_path = r"./data/entities\results\average_score_heatmap.png"
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Average score heatmap saved to: {output_path}")
print(f"Score range: {vmin:.6f} to {vmax:.6f}")
print(f"Color scale range: {vmin_padded:.6f} to {vmax_padded:.6f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

# Directly use the provided statistical data with actual average scores
data = [
    {'Combination': 'method — task', 'Count': 268207, 'Percentage': 26.82, 'Average_Score': 0.451683550944976},
    {'Combination': 'method — method', 'Count': 238326, 'Percentage': 23.83, 'Average_Score': 0.451701455249992},
    {'Combination': 'dataset — method', 'Count': 209839, 'Percentage': 20.98, 'Average_Score': 0.451670681665218},
    {'Combination': 'dataset — task', 'Count': 117049, 'Percentage': 11.70, 'Average_Score': 0.451734064263642},
    {'Combination': 'task — task', 'Count': 74041, 'Percentage': 7.40, 'Average_Score': 0.451716336796281},
    {'Combination': 'dataset — dataset', 'Count': 46223, 'Percentage': 4.62, 'Average_Score': 0.451769339916589},
    {'Combination': 'method — metric', 'Count': 23012, 'Percentage': 2.30, 'Average_Score': 0.451830314605392},
    {'Combination': 'metric — task', 'Count': 12764, 'Percentage': 1.28, 'Average_Score': 0.451561543113404},
    {'Combination': 'dataset — metric', 'Count': 9990, 'Percentage': 1.00, 'Average_Score': 0.451596978053247},
    {'Combination': 'metric — metric', 'Count': 549, 'Percentage': 0.05, 'Average_Score': 0.451556638018898},
]

# Convert to DataFrame
comb_df = pd.DataFrame(data)

# Sort by Count descending
comb_df = comb_df.sort_values('Count', ascending=False).reset_index(drop=True)

# ==================== Score vs Frequency Scatter Plot (Square, Small) ====================
fig, ax = plt.subplots(figsize=(8, 6))

# Create scatter plot
scatter = ax.scatter(comb_df['Count'], comb_df['Average_Score'], 
                     c=comb_df['Average_Score'], cmap='RdYlGn_r', 
                     s=150, alpha=0.7, edgecolors='black', linewidth=1.5)

ax.set_xlabel('Number of Pairs', fontsize=10, fontweight='bold')
ax.set_ylabel('Average Score', fontsize=10, fontweight='bold')
# Code comment
ax.grid(True, alpha=0.3, linestyle='--')

# Add labels for each point with percentage
for _, row in comb_df.iterrows():
    label = f"{row['Combination']}\n({row['Percentage']:.2f}%)"
    ax.annotate(label, (row['Count'], row['Average_Score']),
                xytext=(4, 4), textcoords='offset points', 
                fontsize=7, alpha=0.8, fontweight='semibold',
                ha='left', va='bottom')

# Add statistical information box
correlation = comb_df['Count'].corr(comb_df['Average_Score'])
stats_text = f'Correlation: {correlation:.4f}'
ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
        fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()

# Save the figure
output_path = r"./data/entities\results\score_vs_frequency_scatter.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()